# Using Pushover API to send Notifications

1. Signup for Pushover account (free for 1 month, then $5 for lifetime access)
2. Install Pushover app on your phone or computer
3. Create an Application on Pushover
4. Create an API Key

In [1]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [ ]:
PUSHOVER_USER = os.getenv('PUSHOVER_USER')
PUSHOVER_TOKEN = os.getenv('PUSHOVER_TOKEN')
PUSHOVER_URL = "https://api.pushover.net/1/messages.json"

In [ ]:
def push(message: str):
    print(f"Push: {message}")
    payload = {"user": PUSHOVER_USER, "token": PUSHOVER_TOKEN, "message": message}
    requests.post(PUSHOVER_URL, data=payload)

In [ ]:
def record_user_details(email: str, name: str = "Name not provided", notes: str = "Not provided"):
    push(f"New user registered: {name} ({email}), Notes: {notes}")
    return {"recorded": "ok"}

In [ ]:
def record_unknown_question(question: str):
    push(f"Unknown question given: {question}")
    return {"recorded": "ok"}

In [ ]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [ ]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that was asked, but couldn't be answered"
            }
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [ ]:
tools = [
    {"type": "function", "function": record_user_details_json},
    {"type": "function", "function": record_unknown_question_json}
]

# Print data struct
tools

#### Parse the AI Model's Tool Call Results

```python
# OpenAI Response when it uses a Tool
{
    "choices": [{
        "message": {
            "role": "assistant",
            "content": null,
            "tool_calls": [{
                "id": "call_abc123",
                "function": {
                    "name": "record_user_details",
                    "arguments": '{"email": "john@example.com", "name": "John"}'
                }
            }]
        },
        "finish_reason": "tool_calls"
    }]
}
```

#### 1. Extract the Tool information from the tool calls
```python
tool_name = tool_call.function.name  
arguments = json.loads(tool_call.function.arguments)  
# Input: '{"email": "john@example.com", "name": "John Doe"}'  
# Result: {"email": "john@example.com", "name": "John Doe"}  
```

#### 2. Find the Tool that was called using the tool_name (if the tool exists in the global scope)
```python
tool = globals().get(tool_name)  
```

#### 3. Call the Tool with the arguments
```python
# **arguments unpacks the dictionary into keyword arguments for the function call  

result = tool(**arguments)  
# Becomes: record_user_details(email="john@example.com", name="John Doe")
# Result: {"recorded": "ok", "email_sent": True}
```

#### 4. Format the Response for OpenAI
```python
results.append({  
    "role": "tool",   
    "content": json.dumps(result),  
    "tool_call_id": tool_call.id  
})
```

#### 5. Complete Flow
```python
# Input: AI wants to record user details
tool_calls = [tool_call_object]

# Processing:
# 1. Extract: tool_name = "record_user_details"
# 2. Parse: arguments = {"email": "john@example.com", "name": "John Doe"}  
# 3. Find: tool = record_user_details function
# 4. Execute: record_user_details(email="john@example.com", name="John Doe")
# 5. Get result: {"recorded": "ok", "email_sent": True}

# Output:
results = [{
    "role": "tool",
    "content": '{"recorded": "ok", "email_sent": true}',
    "tool_call_id": "call_abc123"
}]
```

In [ ]:
# Call function by getting the function name (if it's in the global scope)
def handle_tool_calls(tools_calls: list[object]):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name                     # record_user_details
        arguments = json.loads(tool_call.function.arguments)    # {"email": "mail@mail", "name": "John"}
        print(f"Tool called: {tool_name}", flush=True)

        tool = globals().get(tool_name)             # record_user_details()
        result = tool(**arguments) if tool else {}  # record_user_details(email="mail@mail", name="John")
        
        results.append({
            "role": "tool",
            "content": json.dumps(result),          # {"recorded": "ok", "email_sent": True}
            "tool_call_id": tool_call.id
        })

    return results


# Same as above but uses a basic IF Statement to call the function
def handle_tool_calls_simple(tool_calls: list[object]):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name         # record_user_details or record_unknown_question
        arguments = tool_call.function.arguments    # {"email": "mail@mail", "name": "John"} or {"question": "..."}

        if tool_name == "record_user_details":
            result = record_user_details(**arguments)       # record_user_details(email=..., name=...)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)   # record_unknown_question(question=...)

        results.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tool_call.id
        })

In [ ]:
name = "Mark"

with open("../../../resume/resume.md", "r") as f:
    resume = f.read()

with open("../../../resume/summary.txt", "r") as f:
    summary = f.read()

In [ ]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career. \
If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details tool. "

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{resume}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

### Conversation Flow when a Model uses a Tool

#### 1. User Input
```text
Hi, I am interested in your services. My email is john@example.com and my name is John Doe.
```

#### 2. First AI Response
```json
{
    "choices": [{
        "message": {
            "role": "assistant",
            "content": null,
            "tool_calls": [{
                "id": "call_abc123",
                "function": {
                    "name": "record_user_details",
                    "arguments": '{"email": "john@example.com", "name": "John"}'
                }
            }]
        },
        "finish_reason": "tool_calls"
    }]
}
```

#### 3. Tool Execution
```python
# handle_tool_calls() executes and returns a list of dicts
[{
    "role": "tool",
    "content": '{"recorded": "ok", "email_sent": true}',
    "tool_call_id": "call_abc123"
}]
```

#### 4. Update Conversation History after Tool Call
```python
messages = [
    {"role": "system", "content": "You are acting as John Doe..."},
    {"role": "user", "content": "Hi, I'm interested in your services. My email is john@example.com"},
    {"role": "assistant", "content": null, "tool_calls": [...]},  # AI's tool call
    {"role": "tool", "content": '{"recorded": "ok", "email_sent": true}', "tool_call_id": "call_abc123"}
]
```

#### 5. Second Response from the AI
```python
# AI sees the tool result and gives a final response
{
    "choices": [{
        "message": {
            "role": "assistant",
            "content": "Thanks John, I've recorded your details."
        },
        "finish_reason": "stop"
    }]
}
```

In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False

    while not done:
        response = openai.chat.completions.create(
            model="gpt-4o-mini", 
            messages=messages, 
            tools=tools
        )

        finish_reason = response.choices[0].finish_reason

        if finish_reason == "tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True

    return response.choices[0].message.content

## Tool Call Loop

The chat() function uses a While Loop so that when the AI Model makes a Tool Call, it will execute the tool and then continue the conversation.

Without the Loop, the AI Model would only make one Tool Call and then stop without giving an answer.

### 1. First API Call
```python
# Messages sent to AI:
messages = [
    {"role": "system", "content": "You are John Doe..."},
    {"role": "user", "content": "My email is john@example.com"}
]

# AI Response #1:
{
    "finish_reason": "tool_calls",
    "message": {
        "tool_calls": [{"function": {"name": "record_user_details", ...}}]
    }
}
# Loop continues because finish_reason == "tool_calls"
```

### 2. Second API Call
```python
# Messages sent to AI (now includes tool results):
messages = [
    {"role": "system", "content": "You are John Doe..."},
    {"role": "user", "content": "My email is john@example.com"},
    {"role": "assistant", "tool_calls": [...]},  # AI's tool call
    {"role": "tool", "content": '{"recorded": "ok"}', "tool_call_id": "..."}  # Tool result
]

# AI Response #2:
{
    "finish_reason": "stop",
    "message": {
        "content": "Thanks! I've recorded your email and will be in touch."
    }
}
# Loop exits because finish_reason == "stop"
```

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()